# §1 Individual (unfiltered) (v12 full pool subsample 2k, V4 filter) — net of costs

Per-combo metrics and per-combo equity/drawdown curves on the
20% OOS test partition with no ML#2 filter. Sizing: `fixed_dollars_500` (risk $500 per trade, flat).

**Cost model:** every trade is charged `contracts × $5.00` round-trip (≈ $3 retail commission + 2 ticks/side slippage on MNQ at $0.50/tick).

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

REPO = Path.cwd().resolve()
while not (REPO / 'src').exists() and REPO.parent != REPO:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from scripts.evaluation._top_perf_common import (
    STARTING_EQUITY, POLICIES,
    apply_sizing, metrics_from_pnl, monte_carlo,
    load_setup,
    plot_indiv_equity, plot_indiv_dd,
    plot_combined_equity, plot_combined_dd,
    plot_ml2_indiv_equity, plot_ml2_indiv_dd,
    plot_ml2_combined_equity, plot_ml2_combined_dd,
    plot_mc_sims, plot_mc_pnl, plot_mc_sharpe, plot_mc_dd,
)

_ctx = load_setup(cost_per_contract_rt=5.0, top_strategies_path=REPO / 'evaluation' / 'top_strategies_v12_full_pool_2k.json', version='v4')
bars            = _ctx['bars']
YEARS_SPAN      = _ctx['years_span']
strategies      = _ctx['strategies']
results_raw     = _ctx['results_raw']
combined_raw    = _ctx['combined_raw']
combos_ml2      = _ctx['combos_ml2']
s4_pnl_by_combo = _ctx['s4_pnl_by_combo']
ml2_portfolio   = _ctx['ml2_portfolio']


Top-K source: top_strategies_v12_full_pool_2k.json


Test partition: 514,563 bars  2024-10-22 05:08:00 -> 2026-04-08 20:20:00
Years span: 1.461  (used to annualize Sharpe)
Applying friction: $5.00/contract RT (commission + slippage).
Loaded 500 strategies.


Loaded results_raw from cache (500 combos).


Combined unfiltered trades: 1,837,764
Loaded combos_ml2 from cache (500 combos).


ML2 portfolio trade counts: {'fixed_dollars_500': 24542}


In [2]:
rows = []
for r in results_raw:
    if r['trades'].empty:
        for policy in POLICIES:
            rows.append({'combo_id': r['combo_id'], 'policy': policy,
                         **metrics_from_pnl(np.array([]), YEARS_SPAN, policy=policy)})
        continue
    t = r['trades'].sort_values('date', kind='mergesort')
    pnl_base = t['actual_pnl'].to_numpy(dtype=float)
    risk_base = t['dollar_risk'].to_numpy(dtype=float)
    r_mult = np.where(risk_base > 0, pnl_base / risk_base, 0.0)
    for policy in POLICIES:
        pnl = apply_sizing(pnl_base, risk_base, policy)
        rows.append({'combo_id': r['combo_id'], 'policy': policy,
                     **metrics_from_pnl(pnl, YEARS_SPAN, policy=policy, r=r_mult)})
perf1 = pd.DataFrame(rows)
perf1

,combo_id,policy,n_trades,trades_per_year,win_rate,total_pnl_dollars,total_return_pct,sharpe_ratio,max_drawdown_pct,max_drawdown_dollars
0,v11_25,fixed_dollars_500,1005,687.9,0.1990,-86335.81,-172.67,-3.9762,170.74,90075.03
1,v11_68,fixed_dollars_500,1133,775.5,0.3813,-57792.44,-115.58,-2.1615,123.85,62901.96
2,v11_147,fixed_dollars_500,1021,698.8,0.3281,-18650.46,-37.30,-1.0966,54.62,30254.17
3,v11_234,fixed_dollars_500,408,279.3,0.1961,-54624.35,-109.25,-2.5114,110.42,61775.52
4,v11_240,fixed_dollars_500,9169,6275.8,0.1451,-1491250.27,-2982.50,-21.7736,2982.50,1491250.27
...,...,...,...,...,...,...,...,...,...,...
495,v11_29670,fixed_dollars_500,2694,1843.9,0.1344,-260578.25,-521.16,-6.0711,523.60,261799.16
496,v11_29860,fixed_dollars_500,1825,1249.1,0.3677,-226351.92,-452.70,-8.2672,454.81,227406.21
497,v11_29854,fixed_dollars_500,4253,2911.0,0.2807,-497906.45,-995.81,-7.6231,990.10,500460.79
498,v11_29982,fixed_dollars_500,3294,2254.6,0.1409,-634166.19,-1268.33,-15.7313,1271.19,635594.02


In [3]:
plot_indiv_equity(results_raw, 'fixed_dollars_500')

/root/intra/scripts/evaluation/_top_perf_common.py:479: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout(); plt.show()


In [4]:
plot_indiv_dd(results_raw, 'fixed_dollars_500')

/root/intra/scripts/evaluation/_top_perf_common.py:496: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout(); plt.show()
